# AI智能知识检索系统：把文档入库与检索链路拆开学习

这个 Notebook 不启动 HTTP 服务，而是直接调用项目中已经存在的类和函数。目标是把一份文档从 **解析、三级分块、向量化、Milvus 召回、父块回溯，到最终 RAG 检索结果** 分成可以逐格观察的步骤。

> 默认单元只读取本地文件、模型和存储中的已有数据，不会删除或写入 PostgreSQL、Redis、Milvus。请先确认 Milvus、PostgreSQL、Redis 已按项目配置启动。

## 0. 全链路地图

```text
文档文件
  -> DocumentLoader：读取并清洗文本
  -> L1 -> L2 -> L3：建立父子分块树
  -> L1/L2：PostgreSQL（事实来源）+ Redis（缓存）
  -> L3 -> EmbeddingService：每块生成 1024 维 Dense 向量
  -> Milvus：保存 L3、Dense 向量、BM25 稀疏索引和父子 ID

用户问题
  -> 同一 Embedding 模型生成查询向量
  -> Milvus Dense + BM25 Hybrid 召回 L3
  -> 根据 parent_chunk_id / root_chunk_id 回查 L2/L1
  -> Auto-merging + 重排 + 阈值过滤
  -> 返回适合交给聊天模型的上下文
```

后面的单元会依次验证这张图中的每个箭头。

## 1. 运行前准备

在仓库根目录执行下面的命令启动 Notebook：

```powershell
uv run --with jupyterlab jupyter lab
```

如果你只运行到“三级分块”单元，不需要 Milvus、PostgreSQL 或 Redis。运行向量和检索单元时，需要本项目的 `.env` 配置、嵌入模型以及相关服务可用。

In [ ]:
# 找到仓库根目录，并让 Notebook 能导入 backend 包。
from pathlib import Path
import os
import sys


def find_repo_root(start: Path) -> Path:
    """向上查找同时包含 backend 和 pyproject.toml 的目录。"""
    for candidate in (start, *start.parents):
        if (candidate / 'backend').is_dir() and (candidate / 'pyproject.toml').is_file():
            return candidate
    raise RuntimeError('请从AI智能知识检索系统仓库内启动 Jupyter。')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f'仓库根目录：{REPO_ROOT}')
print(f'Python 可导入 backend：{(REPO_ROOT / "backend").is_dir()}')


## 2. 选择一个待学习的文档

此处优先选择本项目已生成的法律测试 Word 文档。没有该文件时，会从 `data/documents/` 或 `output/doc/` 中寻找第一个支持的文件。你也可以手动把 `SAMPLE_PATH` 改成自己的 `.pdf`、`.docx`、`.xlsx` 或 `.html` 文件。

In [ ]:
SUPPORTED_SUFFIXES = {'.pdf', '.doc', '.docx', '.xls', '.xlsx', '.html', '.htm'}
PREFERRED_SAMPLE = 'virtual_legal_knowledge_base_test.docx'

candidates = [
    REPO_ROOT / 'data' / 'documents' / PREFERRED_SAMPLE,
    REPO_ROOT / 'output' / 'doc' / PREFERRED_SAMPLE,
]
for folder in (REPO_ROOT / 'data' / 'documents', REPO_ROOT / 'output' / 'doc'):
    if folder.is_dir():
        candidates.extend(sorted(path for path in folder.iterdir() if path.suffix.lower() in SUPPORTED_SUFFIXES))

SAMPLE_PATH = next((path for path in candidates if path.is_file()), None)
if SAMPLE_PATH is None:
    raise FileNotFoundError('没有找到可解析的示例文件；请手动设置 SAMPLE_PATH。')

print(f'本次示例文件：{SAMPLE_PATH.relative_to(REPO_ROOT)}')
print(f'文件大小：{SAMPLE_PATH.stat().st_size:,} bytes')


## 3. 只做本地解析：从原文件得到 L1 / L2 / L3

`DocumentLoader` 的默认参数为 `chunk_size=800`、`chunk_overlap=100`。它会派生出：

- L1：目标上限约 2400 字符，重叠约 400 字符
- L2：目标上限约 1600 字符，重叠约 200 字符
- L3：目标上限约 800 字符，重叠约 100 字符

这里的长度按字符计算，不是 LLM Token。该单元不连接任何数据库。

In [ ]:
from collections import Counter
from backend.indexing.document_loader import DocumentLoader

loader = DocumentLoader(chunk_size=800, chunk_overlap=100)
all_chunks = loader.load_document(str(SAMPLE_PATH), SAMPLE_PATH.name)

counts = Counter(chunk['chunk_level'] for chunk in all_chunks)
parent_docs = [chunk for chunk in all_chunks if chunk['chunk_level'] in (1, 2)]
leaf_docs = [chunk for chunk in all_chunks if chunk['chunk_level'] == 3]

print(f'全部块数：{len(all_chunks)}')
print(f'L1 根块数：{counts[1]}')
print(f'L2 父块数：{counts[2]}')
print(f'L3 叶子块数：{counts[3]}')
print(f'待写 PostgreSQL/Redis 的父块数：{len(parent_docs)}')
print(f'待写 Milvus 的叶子块数：{len(leaf_docs)}')


### 观察一条块记录

所有块都保留来源信息和父子关系。L1 的 `parent_chunk_id` 为空；L2 指向 L1；L3 指向 L2，同时通过 `root_chunk_id` 直接知道自己属于哪个 L1。

In [ ]:
from pprint import pprint

for chunk in all_chunks[: min(5, len(all_chunks))]:
    print('-' * 88)
    pprint({
        'chunk_id': chunk['chunk_id'],
        'chunk_level': chunk['chunk_level'],
        'parent_chunk_id': chunk['parent_chunk_id'],
        'root_chunk_id': chunk['root_chunk_id'],
        'chunk_idx': chunk['chunk_idx'],
        'text_preview': chunk['text'][:120].replace('\n', ' '),
    })


### 把扁平列表还原成树

`DocumentLoader` 返回的是一个扁平列表，便于批量写存储；下面用 `parent_chunk_id` 将它重新打印成 L1 -> L2 -> L3 的树，验证它不是把全文独立切了三遍。

In [ ]:
from collections import defaultdict

children_by_parent = defaultdict(list)
for chunk in all_chunks:
    children_by_parent[chunk['parent_chunk_id']].append(chunk)

for root in children_by_parent['']:
    print(f"L1  {root['chunk_id']}  ({len(root['text'])} chars)")
    for level_2 in children_by_parent[root['chunk_id']]:
        level_3_children = children_by_parent[level_2['chunk_id']]
        print(f"  L2  {level_2['chunk_id']}  ({len(level_2['text'])} chars)")
        for level_3 in level_3_children:
            print(f"    L3  {level_3['chunk_id']}  ({len(level_3['text'])} chars)")


## 4. 入库时数据会去哪里

真实上传任务会执行下面的分流：

```python
parent_chunk_store.upsert_documents(parent_docs)  # L1/L2 -> PostgreSQL；提交后尝试写 Redis
milvus_writer.write_documents(leaf_docs)           # L3 -> Embedding -> Milvus
```

此 Notebook 只构造一条即将写入 Milvus 的数据预览，不执行写入。注意 `dense_embedding` 现在还没有生成。

In [ ]:
if not leaf_docs:
    raise RuntimeError('当前文档没有生成 L3，无法继续演示向量化。')

leaf = leaf_docs[0]
milvus_row_preview = {
    'dense_embedding': '<下一单元生成的 1024 个 float>',
    'text': leaf['text'][:160] + ('...' if len(leaf['text']) > 160 else ''),
    'filename': leaf['filename'],
    'file_type': leaf['file_type'],
    'chunk_id': leaf['chunk_id'],
    'parent_chunk_id': leaf['parent_chunk_id'],
    'root_chunk_id': leaf['root_chunk_id'],
    'chunk_level': leaf['chunk_level'],
}
pprint(milvus_row_preview)


## 5. 对一条 L3 生成 Dense 向量

项目配置的 `BAAI/bge-m3` 原生输出是 1024 维，因此 `DENSE_EMBEDDING_DIM` 与 Milvus 的 `dense_embedding` schema 都必须设为 1024。

这一格会首次加载本地 Embedding 模型，CPU 环境下可能需要一些时间；它只在内存中计算，不写 Milvus。

In [ ]:
from math import sqrt
from backend.indexing.embedding import embedding_service

vector = embedding_service.get_embeddings([leaf['text']])[0]
vector_norm = sqrt(sum(value * value for value in vector))

print(f'向量维度：{len(vector)}')
print(f'向量范数：{vector_norm:.6f}（归一化后应接近 1）')
print(f'前 12 个浮点数：{vector[:12]}')
assert len(vector) == 1024, '模型输出维度与预期不一致。'


## 6. 检索的第一层：问题 -> 查询向量 -> Milvus 原始 Hybrid 召回

这一节假设目标文件已经通过项目上传接口进入 Milvus。Hybrid 召回同时使用：

- Dense：查询和 L3 文本的语义向量相似度。
- BM25：Milvus 根据同一个 `text` 字段自动维护的关键词稀疏索引。

这里展示的是**原始 L3 候选**，还没有执行父块合并、重排和阈值过滤。

In [ ]:
from backend.indexing.milvus_client import get_milvus_store

query = '违约方收到通知后需要在多长时间内完成整改？'
query_vector = embedding_service.get_embeddings([query])[0]
milvus_store = get_milvus_store()
milvus_store.init_collection(len(query_vector))

raw_hits = milvus_store.hybrid_retrieve(
    dense_embedding=query_vector,
    query=query,
    top_k=6,
    filter_expr='chunk_level == 3',
)

print(f'问题：{query}')
print(f'原始 L3 候选数：{len(raw_hits)}')
for rank, hit in enumerate(raw_hits, start=1):
    preview = (hit.get('text') or '')[:120].replace('\n', ' ')
    print(f"{rank}. {hit.get('chunk_id')} | parent={hit.get('parent_chunk_id')}")
    print(f"   {preview}")


## 7. 手动观察一次父块回查

原始命中是 L3。它的 `parent_chunk_id` 指向 L2，`root_chunk_id` 指向 L1。`ParentChunkStore.get_documents_by_ids()` 会先查 Redis；缓存未命中时再查 PostgreSQL。

这正是本项目不把 L1/L2 也塞进 Milvus 的原因：L3 用来精确定位，父块用来补全回答上下文。

In [ ]:
from backend.indexing.parent_chunk_store import ParentChunkStore

if not raw_hits:
    print('没有原始命中；请确认 Milvus 已启动且目标文档已入库。')
else:
    first_hit = raw_hits[0]
    ancestor_ids = [
        chunk_id
        for chunk_id in [first_hit.get('parent_chunk_id'), first_hit.get('root_chunk_id')]
        if chunk_id
    ]
    parent_store = ParentChunkStore()
    ancestors = parent_store.get_documents_by_ids(ancestor_ids)

    print(f"命中 L3：{first_hit.get('chunk_id')}")
    for ancestor in ancestors:
        preview = (ancestor.get('text') or '')[:160].replace('\n', ' ')
        print(f"L{ancestor.get('chunk_level')} {ancestor.get('chunk_id')}：{preview}")


## 8. 调用项目的完整检索函数

`backend.rag.utils.retrieve_documents()` 负责完整流水线：

```text
查询向量
  -> Hybrid 召回（失败时降级为 Dense）
  -> Auto-merging：按父子关系恢复更大上下文
  -> 可选重排
  -> 最低分阈值过滤
  -> docs + meta
```

`meta` 是学习这条链路最有用的诊断信息：它记录了实际使用的检索模式、候选数量、合并情况和重排情况。

In [ ]:
from backend.rag.utils import retrieve_documents

result = retrieve_documents(query, top_k=3)
docs = result['docs']
meta = result['meta']

interesting_meta_keys = [
    'retrieval_mode', 'retrieval_pipeline', 'candidate_k', 'recall_count',
    'merged_parent_count', 'rerank_applied', 'post_rerank_count',
    'post_threshold_count', 'retrieval_empty',
]
print('检索诊断：')
for key in interesting_meta_keys:
    print(f'  {key}: {meta.get(key)}')

print('\n最终交给 RAG/Agent 的上下文块：')
for rank, doc in enumerate(docs, start=1):
    preview = (doc.get('text') or '')[:220].replace('\n', ' ')
    print(f"{rank}. level={doc.get('chunk_level')} | {doc.get('chunk_id')}")
    print(f"   {preview}")


## 9. 建议的练习

1. 将 `DocumentLoader(chunk_size=800, chunk_overlap=100)` 改为 `chunk_size=500`，比较 L1/L2/L3 数量与重叠带来的重复。
2. 更换 `query`：分别尝试精确术语、同义表达和与文档无关的问题，观察 Hybrid 的效果。
3. 比较第 6 节的 `raw_hits` 与第 8 节的 `docs`：前者是小 L3，后者可能已经被父块上卷。
4. 暂时停止 Redis 后重复第 7 节，理解缓存未命中时 PostgreSQL 如何成为父块事实来源。
5. 阅读 `backend/api/routes/documents.py` 中 `_process_upload_job()`，把 Notebook 里的数据流对应回真实 HTTP 上传与进度轮询。

不要直接在生产 collection 上用同名文件做写入实验。当前项目以 `filename` 作为清理范围，重传同名文件会先删除旧索引。